# Koschei Sentinel — Kaggle CPU-Only Cyber SFT Preflight

Run this notebook with **Accelerator = None** and **Internet = ON**. It does not perform training and does not load model weights. It verifies exact Qwen3.5 model revisions, the final Transformers runtime floor, Defense Reflex v3 planning, chat-template/tokenizer compatibility, and sequence-length fit for micro, 9B normal, and 9B low-memory configs.

Only move to the GPU micro-first notebook when the final summary reports `ready_for_gpu_micro_attempt=true`.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/koschei-sentinel')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/bugsbuny243/koschei-sentinel.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess
env = os.environ.copy()
env['KOSCHEI_KAGGLE_CPU_PREFLIGHT_ROOT'] = '/kaggle/working/koschei-sentinel-cpu-preflight'
subprocess.run(
    ['bash', str(repo / 'scripts/preflight_cyber_sft_qwen35_kaggle_cpu.sh'), str(repo)],
    check=True,
    env=env,
)


In [ ]:
import json
from pathlib import Path

out = Path('/kaggle/working/koschei-sentinel-cpu-preflight')
summary = json.loads((out / 'cpu-preflight-summary.json').read_text())
assert summary['gpu_used'] is False
assert summary['model_weights_loaded'] is False
assert summary['ready_for_gpu_micro_attempt'] is True
print('Transformers:', summary['transformers_version'])
for report in summary['reports']:
    print('\nConfig:', report['config_path'])
    print('Static plan ready:', report['static_plan_ready'])
    print('Tokenization ready:', report['tokenization_ready'])
    print('Max observed tokens:', report['max_observed_sequence_tokens'])
    print('Configured max tokens:', report['plan']['max_sequence_length'] if 'max_sequence_length' in report['plan'] else 'see config')
print('\nCPU preflight passed. GPU micro-smoke may now be attempted.')
